In [1]:
import cProfile

from poisson_hypergraph import GH
from NMI_func import NMI
import xgi
import numpy as np
import networkx as nx
import math
from numba import jit, njit

# function that generates an intstance of the class GH with the parameters stored in true_theta and for timesteps times resulting in a graph with timesteps+1 edges
# the graph starts with 2 nodes but will have more as novel nodes are added
def generate_graph(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0, 1]])
    H.set_node_attributes({0 : 0, 1 : 1}, name = "label")
    g = GH(H, [0, 1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)
    return g

def generate_graph_26_starting_nodes(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]])
    H.set_node_attributes({0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:1,14:1,15:1,16:1,17:1,18:1,19:1,20:1,21:1,22:1,23:1,24:1,25:1}, name="label")
    g=GH(H, [0,1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)

    return g

true_p = .9
true_q = .1
# gamma_nu is poisson weight for the distribution for number of like labeled novel nodes of choosen u novel nodes added
# gamma_nr is poisson weight for the distribution for number of opposite labeled nodes of choosen u novel nodes added
# gamma_eu is poisson weight for the distribution for number of like labeled nodes of choosen u external nodes added
# gamma_er is poisson weight for the distribution for number of opposite labeled nodes of choosen u external nodes added
gamma_nu, gamma_nr, gamma_eu, gamma_er = .01, .01, 1, 0.25

# should be in this order
true_theta = [true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er]
timesteps = 10

g = generate_graph_26_starting_nodes(true_theta, timesteps)

# input is an instance of the class GH defined in poisson_hypergraph.py... output a tuple of lists of total likelihood and nmi indexed by timestep
def greedy_community_detection_algo(g, generate_likelihoods):
    total_likelihoods = []
    nmis = []

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 1000
    step_num = 0
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(1, len(g.get_edges())))
        
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)


        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) / len(canidate_f)
        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
            # print("swapped label of " + str(v_index) + " from edge " + str(e_index))

        step_num+=1

        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))
        
    # print("greedy labels likelihood: " + str(g.expected_log_likelihood_total(true_theta, null_labels)))
    # print("true labels likelihood: " + str(g.expected_log_likelihood_total(true_theta,true_labels)))
    # print(null_labels)
    # print(true_labels)
    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_with_posterior_prob(g, generate_likelihoods):
    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        e_index = np.random.choice(range(0, len(g.get_edges())))
        v_index = np.random.choice(list(g.get_edges()[e_index]))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

        canidate_f = []
        for f_index in range(e_index):
            if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                canidate_f.append(f_index)

        for f_index in canidate_f:
            delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        
        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

def greedy_community_detection_algo_3_label(g, generate_likelihoods):
    NUM_ES = 3

    null_labels = np.random.choice([0,1], size=len(g.get_labels()))
    true_labels = g.get_labels()

    greedy_steps = 2000
    step_num = 0

    # TODO remove, for testing
    total_likelihoods = []
    nmis = []
    while step_num < greedy_steps:
        delta_E = 0
        
        v_index = np.random.choice(list(range(len(g.get_labels()))))

        canidate_e = []
        for e_index in range(1, len(g.get_edges())):
            if v_index in g.get_edges()[e_index]:
                canidate_e.append(e_index)

        e_indexes = np.random.choice(canidate_e, min(NUM_ES, len(canidate_e)))

        new_labels = null_labels.copy()
        new_labels[v_index] = 1 - new_labels[v_index]

        for e_index in e_indexes:
            f_probs = g.f_prob_array_given_e(e_index, true_theta, new_labels)

            canidate_f = []
            for f_index in range(e_index):
                if len(g.get_edges()[f_index].intersection(g.get_edges()[e_index])) != 0:
                    canidate_f.append(f_index)

            for f_index in canidate_f:
                delta_E += g.greedy_expectation_step_given_f(v_index, f_index, e_index, true_theta, null_labels) * f_probs[f_index]


        if (delta_E > 0):
            null_labels[v_index] = 1 - null_labels[v_index]
        
        # if (step_num % 25 == 0):
        if generate_likelihoods:
            total_likelihoods.append(g.total_log_likelihood(true_theta, null_labels))
            nmis.append(NMI(g.get_labels(), null_labels, g))

            
        step_num+=1

    if generate_likelihoods:
        return total_likelihoods, nmis
    else:
        return null_labels

In [2]:
timesteps = 100
def test():
    for _ in range(1):
        g = generate_graph(true_theta, timesteps)
        # False specifies whether to generate true log likelihood every step for graphing

        # both lines below call the different versions of the algo... 
        # 3_label has been the best in testing but takes longer



        # greedy_community_detection_algo_with_posterior_prob(g, False)
        greedy_community_detection_algo_3_label(g, False)


cProfile.run('test()', sort='ncalls')

         31334899 function calls (31334898 primitive calls) in 25.957 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  6483221    1.443    0.000    1.443    0.000 {built-in method builtins.sum}
5795658/5795657    0.470    0.000    0.470    0.000 {built-in method builtins.len}
  4485258    0.879    0.000    0.879    0.000 {built-in method math.log}
  3009033    0.338    0.000    0.339    0.000 {built-in method builtins.isinstance}
  2990172    1.210    0.000    1.546    0.000 poisson_hypergraph.py:914(gammaln_fast)
  1842584    0.185    0.000    0.185    0.000 poisson_hypergraph.py:28(get_edges)
  1787746    0.335    0.000    0.335    0.000 {method 'intersection' of 'set' objects}
   696726    0.077    0.000    0.077    0.000 {method 'append' of 'list' objects}
   309978    0.181    0.000    0.181    0.000 poisson_hypergraph.py:413(<listcomp>)
   309978    0.131    0.000    0.131    0.000 poisson_hypergraph.py:415(<listcomp>)


In [3]:
print(g.f_prob_array_given_e_array(3, true_theta, g.get_labels()))

print(g.get_edges()[3])
print(g.get_edges()[0])
print(g.get_edges()[1])
print(g.get_edges()[2])

[0.00000000e+00 3.05999787e-15 1.00000000e+00]
{13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25}
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25}
{4, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25}
{14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25}


In [8]:
k = np.random.choice([201, 202], 100)
k2 = np.random.choice([199, 198], 100)

method1 = lambda k: g.gammaln_fast(k)
method2 = lambda k: g.gammaln_fast(k2)

%timeit method1(k)
%timeit method2(k)

2.73 μs ± 13.7 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
2.71 μs ± 8.47 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [5]:

k = 0

method1 = lambda k: g.f_prob_array_given_e(3, true_theta, g.get_labels())
method2 = lambda k: g.f_prob_array_given_e_array(3, true_theta, g.get_labels())

def test2():
    for _ in range(100000):
        g.f_prob_array_given_e_array(8, true_theta, g.get_labels())

%timeit method1(k)
%timeit method2(k)

cProfile.run('test2()', sort='ncalls')

72.6 μs ± 523 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
161 μs ± 1.53 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
         34400004 function calls in 29.393 seconds

   Ordered by: call count

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  8200000    1.540    0.000    1.540    0.000 {built-in method builtins.sum}
  8200000    0.651    0.000    0.651    0.000 {method 'append' of 'list' objects}
  4100000    0.309    0.000    0.309    0.000 {built-in method builtins.len}
  1800000    0.389    0.000    0.389    0.000 {built-in method math.log}
  1600000    0.489    0.000    0.489    0.000 {method 'intersection' of 'set' objects}
  1500000    0.152    0.000    0.152    0.000 {built-in method builtins.isinstance}
  1200000    1.108    0.000    1.108    0.000 {built-in method numpy.array}
  1200000    2.124    0.000    2.230    0.000 poisson_hypergraph.py:914(gammaln_fast)
   800000    0.321    0.000    0.321    0.000 poisson_hyper

In [11]:
def test2():
    for _ in range(100000):
        g.f_prob_array_given_e_array(8, true_theta, g.get_labels())

cProfile.run('test2()', sort='tottime')

         34400004 function calls in 29.413 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
   100000   15.820    0.000   29.012    0.000 poisson_hypergraph.py:767(f_prob_array_given_e_array)
  1200000    2.138    0.000    2.244    0.000 poisson_hypergraph.py:914(gammaln_fast)
  8200000    1.536    0.000    1.536    0.000 {built-in method builtins.sum}
  1200000    1.114    0.000    1.114    0.000 {built-in method numpy.array}
   100000    0.949    0.000    0.949    0.000 poisson_hypergraph.py:797(<listcomp>)
   100000    0.684    0.000    2.377    0.000 _logsumexp.py:250(softmax)
   800000    0.681    0.000    0.681    0.000 poisson_hypergraph.py:788(<listcomp>)
  8200000    0.660    0.000    0.660    0.000 {method 'append' of 'list' objects}
   800000    0.601    0.000    0.601    0.000 poisson_hypergraph.py:827(<listcomp>)
   200000    0.523    0.000    0.523    0.000 {method 'reduce' of 'numpy.ufunc' objects}
   800000  

In [ ]:
L = range(10000)

def method_1(): 
    L2 = []
    for i in L:
        L2.append(i**2)
    return np.array(L2)

def method_2(): 
    L2 = np.zeros(len(L))
    for i in range(len(L)):
        L2[i] = L[i]**2
    return L2

def method_3():
    return L**2

%timeit method_1()
%timeit method_2()


3.02 ms ± 5.49 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
3.95 ms ± 10.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
